<a href="https://colab.research.google.com/github/lizlorenzana931109-art/FinPredict-Actividad5/blob/main/fuentes/03_modelado_experimentacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# FinPredict | Actividad 5
## 03. Modelado y experimentación

### Objetivo

Desarrollar, optimizar y comparar modelos supervisados de Machine Learning para predecir el incumplimiento de pago hipotecario a 90 días.

### Modelos seleccionados

1. **Regresión logística:** modelo de referencia interpretable.
2. **Random Forest:** modelo de ensamble capaz de identificar relaciones no lineales.

### Metodología

- Carga del dataset preparado `v1`.
- Separación temporal entre desarrollo y evaluación final.
- Construcción de matrices predictoras.
- Entrenamiento de modelos base.
- Optimización mediante GridSearchCV.
- Registro de experimentos con MLflow.
- Comparación de métricas y selección del modelo.
- Evaluación final sobre el periodo reservado.

### Métricas

Se evaluarán:

- Precision.
- Recall.
- F1-score.
- ROC-AUC.
- PR-AUC.
- Matriz de confusión.

En riesgo crediticio, prestaremos especial atención al **recall de incumplimiento**, sin ignorar el costo operativo de los falsos positivos.

### Controles metodológicos

- Utilizar únicamente las 14 variables predictoras seleccionadas.
- Excluir identificadores y columnas de auditoría del entrenamiento.
- Mantener reservada la evaluación final.
- Evitar fuga de información durante el preprocesamiento y la validación temporal.
- Documentar parámetros, resultados y decisiones de selección.

**Proyecto:** FinPredict  
**Etapa:** Fase 2 — Modelado y experimentación  
**Herramientas:** Python, scikit-learn, MLflow y Google Colab


In [1]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 2
# Carga y verificación del conjunto de desarrollo
# ============================================================

import pandas as pd
import numpy as np
import requests
import hashlib
from io import BytesIO

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "lizlorenzana931109-art/"
    "FinPredict-Actividad5/main/datos/datos_limp/"
)

# Descargar manifiesto de la versión v1
respuesta = requests.get(
    BASE_URL + "manifiesto_v1.json",
    timeout=60
)
respuesta.raise_for_status()

manifiesto = respuesta.json()

# Descargar únicamente el conjunto de desarrollo
nombre_archivo = "finpredict_desarrollo_v1.parquet"

respuesta = requests.get(
    BASE_URL + nombre_archivo,
    timeout=180
)
respuesta.raise_for_status()

contenido = respuesta.content

# Validar integridad contra el manifiesto
sha256 = hashlib.sha256(contenido).hexdigest()

assert sha256 == manifiesto["archivos"][nombre_archivo]["sha256"], (
    "La huella SHA-256 no coincide."
)

# Cargar datos
df_desarrollo = pd.read_parquet(BytesIO(contenido))

# Recuperar selección de predictores
PREDICTORES = manifiesto["predictores"]
VARIABLE_OBJETIVO = manifiesto["variable_objetivo"]

# Controles básicos
assert len(df_desarrollo) == 350_520
assert len(PREDICTORES) == 14
assert df_desarrollo["uso"].eq("HISTORICO").all()
assert not df_desarrollo[PREDICTORES].isna().any().any()

print("FINPREDICT | Carga del conjunto de desarrollo")
print("-" * 60)

print(f"Observaciones: {len(df_desarrollo):,}")
print(f"Columnas: {df_desarrollo.shape[1]}")
print(f"Predictores: {len(PREDICTORES)}")
print(f"Eventos de incumplimiento: {df_desarrollo[VARIABLE_OBJETIVO].sum():,}")
print(f"Integridad SHA-256: VERIFICADA")

print("-" * 60)
print("Datos de desarrollo cargados correctamente.")


FINPREDICT | Carga del conjunto de desarrollo
------------------------------------------------------------
Observaciones: 350,520
Columnas: 34
Predictores: 14
Eventos de incumplimiento: 21,368
Integridad SHA-256: VERIFICADA
------------------------------------------------------------
Datos de desarrollo cargados correctamente.


In [2]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 3
# Definición de validación temporal expansiva
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import PredefinedSplit

# Ordenar el histórico sin modificar la fuente original
df_modelado = (
    df_desarrollo
    .sort_values(["fecha_corte", "id_credito"])
    .reset_index(drop=True)
)

# Cortes históricos disponibles
cortes = sorted(df_modelado["fecha_corte"].unique())

# Elegimos tres ventanas de validación
cortes_validacion = cortes[-3:]

particiones = []

for corte_validacion in cortes_validacion:

    # Solo entrenar con observaciones cuyas
    # etiquetas ya eran conocidas al corte de validación
    mascara_train = (
        df_modelado["fin_prediccion"] < corte_validacion
    )

    mascara_validacion = (
        df_modelado["fecha_corte"] == corte_validacion
    )

    idx_train = np.flatnonzero(mascara_train)
    idx_validacion = np.flatnonzero(mascara_validacion)

    assert len(idx_train) > 0
    assert len(idx_validacion) > 0

    assert (
        df_modelado.loc[
            idx_train, "fin_prediccion"
        ].max()
        < corte_validacion
    )

    particiones.append((idx_train, idx_validacion))

# Resumen de particiones
resumen_particiones = pd.DataFrame([
    {
        "fold": i,
        "ultimo_corte_train": df_modelado.loc[
            idx_train, "fecha_corte"
        ].max().date(),
        "corte_validacion": df_modelado.loc[
            idx_validacion, "fecha_corte"
        ].iloc[0].date(),
        "registros_train": len(idx_train),
        "registros_validacion": len(idx_validacion),
        "eventos_validacion": int(
            df_modelado.loc[
                idx_validacion, VARIABLE_OBJETIVO
            ].sum()
        )
    }
    for i, (idx_train, idx_validacion)
    in enumerate(particiones, start=1)
])

print("FINPREDICT | Validación temporal")
print("-" * 65)

display(resumen_particiones)

print("-" * 65)
print("Particiones temporales construidas.")
print("Evaluación final 2025-Q3 permanece reservada.")


FINPREDICT | Validación temporal
-----------------------------------------------------------------


,fold,ultimo_corte_train,corte_validacion,registros_train,registros_validacion,eventos_validacion
0,1,2024-06-30,2024-12-31,210863,33792,1848
1,2,2024-09-30,2025-03-31,242453,36111,2116
2,3,2024-12-31,2025-06-30,276245,38164,2252


-----------------------------------------------------------------
Particiones temporales construidas.
Evaluación final 2025-Q3 permanece reservada.


In [3]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 4
# Preparación de entradas para modelado
# ============================================================

import numpy as np
import pandas as pd

# Conservar intacto el dataset cargado
df_entrenamiento = df_modelado.copy(deep=True)

# Identificar ingresos imputados estadísticamente en v1
mascara_mediana = (
    df_entrenamiento["metodo_recuperacion_ingreso"]
    == "MEDIANA_TIPO_EMPLEO"
)

registros_mediana = int(mascara_mediana.sum())

# Retirar la imputación estadística global
df_entrenamiento.loc[
    mascara_mediana, "ingreso_mensual"
] = np.nan

# Retirar también la relación derivada de esos ingresos
df_entrenamiento.loc[
    mascara_mediana, "relacion_pago_ingreso"
] = np.nan

# Construir matrices sin incluir auditoría ni identificadores
X = df_entrenamiento[PREDICTORES].copy()

y = df_entrenamiento[VARIABLE_OBJETIVO].copy()

# Validaciones
assert len(X) == len(y) == 350_520

assert X["ingreso_mensual"].isna().sum() == registros_mediana
assert X["relacion_pago_ingreso"].isna().sum() == registros_mediana

assert y.isin([0, 1]).all()

print("FINPREDICT | Preparación para modelado")
print("-" * 60)

print(f"Observaciones: {len(X):,}")
print(f"Predictores: {X.shape[1]}")
print(f"Ingresos pendientes de imputación por fold: {registros_mediana}")
print(f"Relaciones pago-ingreso pendientes: {X['relacion_pago_ingreso'].isna().sum():,}")

print("-" * 60)
print("Matrices preparadas. No se ha entrenado ningún modelo.")


FINPREDICT | Preparación para modelado
------------------------------------------------------------
Observaciones: 350,520
Predictores: 14
Ingresos pendientes de imputación por fold: 41
Relaciones pago-ingreso pendientes: 41
------------------------------------------------------------
Matrices preparadas. No se ha entrenado ningún modelo.



### 5. Pipeline de preprocesamiento

El preprocesamiento se integrará a los modelos mediante `Pipeline` de scikit-learn.

Esto permitirá ajustar los parámetros de imputación exclusivamente con los registros disponibles en cada conjunto de entrenamiento durante la validación temporal.

La transformación contempla la dependencia matemática entre `ingreso_mensual` y `relacion_pago_ingreso`, recalculando esta última cuando el ingreso sea imputado.

Para la regresión logística se aplicará posteriormente estandarización de variables mediante `StandardScaler`. Para Random Forest no será necesaria.


In [4]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 5
# Transformador de imputación y consistencia financiera
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


class ImputadorFinPredict(BaseEstimator, TransformerMixin):
    """
    Imputa ingreso_mensual con la mediana calculada
    durante fit y recalcula relacion_pago_ingreso
    únicamente para los registros imputados.
    """

    def fit(self, X, y=None):

        X = X.copy()

        self.mediana_ingreso_ = X[
            "ingreso_mensual"
        ].median()

        if pd.isna(self.mediana_ingreso_):
            raise ValueError(
                "No hay ingresos válidos para calcular la mediana."
            )

        if self.mediana_ingreso_ <= 0:
            raise ValueError(
                "La mediana del ingreso debe ser positiva."
            )

        return self

    def transform(self, X):

        X = X.copy()

        # Identificar únicamente los ingresos faltantes
        mascara_nulos = X["ingreso_mensual"].isna()

        # Aplicar mediana aprendida en fit
        X["ingreso_mensual"] = (
            X["ingreso_mensual"]
            .fillna(self.mediana_ingreso_)
        )

        # Recalcular relación financiera de esos registros
        X.loc[
            mascara_nulos, "relacion_pago_ingreso"
        ] = (
            X.loc[mascara_nulos, "pago_mensual"]
            / X.loc[mascara_nulos, "ingreso_mensual"]
        )

        # Verificar ausencia de nulos e infinitos
        if X.isna().any().any():
            raise ValueError(
                "Persisten valores faltantes después del tratamiento."
            )

        if not np.isfinite(
            X.to_numpy(dtype=float)
        ).all():
            raise ValueError(
                "Se encontraron valores infinitos."
            )

        return X


# Probar el transformador sin entrenar modelos
transformador_prueba = ImputadorFinPredict()

# Solo para verificar funcionamiento:
# el ajuste definitivo se hará dentro de cada fold
X_prueba = transformador_prueba.fit_transform(X)

assert X_prueba.isna().sum().sum() == 0

assert np.isfinite(
    X_prueba.to_numpy(dtype=float)
).all()

assert len(X_prueba) == len(X)

print("FINPREDICT | Preprocesamiento")
print("-" * 60)

print(
    f"Mediana de prueba: "
    f"${transformador_prueba.mediana_ingreso_:,.2f}"
)

print(
    f"Valores faltantes restantes: "
    f"{X_prueba.isna().sum().sum()}"
)

print(f"Variables conservadas: {X_prueba.shape[1]}")

print("-" * 60)
print("Transformador validado.")
print("Los modelos todavía no han sido entrenados.")


FINPREDICT | Preprocesamiento
------------------------------------------------------------
Mediana de prueba: $85,978.11
Valores faltantes restantes: 0
Variables conservadas: 14
------------------------------------------------------------
Transformador validado.
Los modelos todavía no han sido entrenados.



### 6. Modelo base: regresión logística

Se entrena un modelo de regresión logística como referencia inicial para la predicción del incumplimiento hipotecario.

El modelo utiliza un Pipeline que integra:

1. Imputación de ingresos y conciliación de la relación pago-ingreso.
2. Estandarización de variables numéricas.
3. Regresión logística con ponderación de clases.

El primer experimento utiliza una partición temporal y se evalúa mediante Precision, Recall, F1-score, ROC-AUC y PR-AUC.

Esta evaluación es preliminar. La selección definitiva del modelo requerirá validación temporal y optimización de hiperparámetros.


In [5]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 6
# Regresión logística | Modelo base
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

import pandas as pd
import numpy as np

# 1. Primera partición temporal
idx_train, idx_validacion = particiones[0]

X_train = X.iloc[idx_train].copy()
y_train = y.iloc[idx_train].copy()

X_val = X.iloc[idx_validacion].copy()
y_val = y.iloc[idx_validacion].copy()

# 2. Pipeline completo
pipeline_logistico = Pipeline([
    ("imputacion", ImputadorFinPredict()),
    ("escalado", StandardScaler()),
    ("modelo", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        random_state=2028
    ))
])

# 3. Entrenamiento
pipeline_logistico.fit(X_train, y_train)

# 4. Predicciones sobre validación
probabilidades = pipeline_logistico.predict_proba(X_val)[:, 1]

# Umbral inicial de clasificación
UMBRAL_BASE = 0.50

predicciones = (
    probabilidades >= UMBRAL_BASE
).astype(int)

# 5. Evaluación
metricas_logistica_base = {
    "Precision": precision_score(
        y_val, predicciones, zero_division=0
    ),
    "Recall": recall_score(
        y_val, predicciones, zero_division=0
    ),
    "F1-score": f1_score(
        y_val, predicciones, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val, probabilidades
    ),
    "PR-AUC": average_precision_score(
        y_val, probabilidades
    )
}

resumen_metricas = pd.DataFrame(
    metricas_logistica_base.items(),
    columns=["metrica", "valor"]
)

matriz_confusion = confusion_matrix(
    y_val,
    predicciones,
    labels=[0, 1]
)

print("FINPREDICT | Regresión logística - Modelo base")
print("-" * 65)

print(f"Entrenamiento: {len(X_train):,}")
print(f"Validación: {len(X_val):,}")
print(f"Incumplimientos reales: {int(y_val.sum()):,}")
print(f"Umbral: {UMBRAL_BASE:.2f}")

print("\nMétricas de validación:")
display(resumen_metricas.round(4))

print("\nMatriz de confusión:")
display(pd.DataFrame(
    matriz_confusion,
    index=["Real: sin incumplimiento", "Real: incumplimiento"],
    columns=[
        "Predicho: sin incumplimiento",
        "Predicho: incumplimiento"
    ]
))

print("-" * 65)
print("Primer entrenamiento completado.")
print("Evaluación final reservada sin utilizar.")


FINPREDICT | Regresión logística - Modelo base
-----------------------------------------------------------------
Entrenamiento: 210,863
Validación: 33,792
Incumplimientos reales: 1,848
Umbral: 0.50

Métricas de validación:


,metrica,valor
0,Precision,0.5217
1,Recall,0.9351
2,F1-score,0.6698
3,ROC-AUC,0.9859
4,PR-AUC,0.8547



Matriz de confusión:


,Predicho: sin incumplimiento,Predicho: incumplimiento
Real: sin incumplimiento,30360,1584
Real: incumplimiento,120,1728


-----------------------------------------------------------------
Primer entrenamiento completado.
Evaluación final reservada sin utilizar.


In [6]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 7
# Random Forest | Modelo base
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
import pandas as pd
import time

# 1. Pipeline del modelo
# Random Forest no requiere estandarización.
pipeline_rf = Pipeline([
    ("imputacion", ImputadorFinPredict()),
    ("modelo", RandomForestClassifier(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        n_jobs=-1,
        random_state=2028
    ))
])

# 2. Entrenamiento: mismo fold que regresión logística
inicio = time.perf_counter()

pipeline_rf.fit(X_train, y_train)

tiempo_rf = time.perf_counter() - inicio

# 3. Predicciones
probabilidades_rf = pipeline_rf.predict_proba(X_val)[:, 1]

predicciones_rf = (
    probabilidades_rf >= UMBRAL_BASE
).astype(int)

# 4. Métricas
metricas_rf_base = {
    "Precision": precision_score(
        y_val, predicciones_rf, zero_division=0
    ),
    "Recall": recall_score(
        y_val, predicciones_rf, zero_division=0
    ),
    "F1-score": f1_score(
        y_val, predicciones_rf, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val, probabilidades_rf
    ),
    "PR-AUC": average_precision_score(
        y_val, probabilidades_rf
    )
}

# 5. Comparación inicial de modelos
comparativo_base = pd.DataFrame({
    "Regresión logística": metricas_logistica_base,
    "Random Forest": metricas_rf_base
}).round(4)

matriz_rf = confusion_matrix(
    y_val,
    predicciones_rf,
    labels=[0, 1]
)

print("FINPREDICT | Random Forest - Modelo base")
print("-" * 65)

print(f"Entrenamiento: {len(X_train):,}")
print(f"Validación: {len(X_val):,}")
print(f"Tiempo de entrenamiento: {tiempo_rf:.2f} segundos")

print("\nComparación inicial:")
display(comparativo_base)

print("\nMatriz de confusión - Random Forest:")
display(pd.DataFrame(
    matriz_rf,
    index=[
        "Real: sin incumplimiento",
        "Real: incumplimiento"
    ],
    columns=[
        "Predicho: sin incumplimiento",
        "Predicho: incumplimiento"
    ]
))

print("-" * 65)
print("Segundo modelo base entrenado.")
print("Evaluación final permanece reservada.")


FINPREDICT | Random Forest - Modelo base
-----------------------------------------------------------------
Entrenamiento: 210,863
Validación: 33,792
Tiempo de entrenamiento: 66.55 segundos

Comparación inicial:


,Regresión logística,Random Forest
Precision,0.5217,0.5300
Recall,0.9351,0.9324
F1-score,0.6698,0.6758
ROC-AUC,0.9859,0.9860
PR-AUC,0.8547,0.8650



Matriz de confusión - Random Forest:


,Predicho: sin incumplimiento,Predicho: incumplimiento
Real: sin incumplimiento,30416,1528
Real: incumplimiento,125,1723


-----------------------------------------------------------------
Segundo modelo base entrenado.
Evaluación final permanece reservada.


In [7]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 8
# Verificación del esquema de validación para GridSearchCV
# ============================================================

import numpy as np
import pandas as pd

# Verificar los tres folds temporales
registros_validacion = []

for numero, (idx_train, idx_val) in enumerate(
    particiones, start=1
):

    train = df_modelado.iloc[idx_train]
    val = df_modelado.iloc[idx_val]

    # Cada validación debe utilizar un corte distinto
    assert val["fecha_corte"].nunique() == 1

    # Las etiquetas de entrenamiento deben estar
    # disponibles antes del corte de validación
    assert (
        train["fin_prediccion"].max()
        < val["fecha_corte"].min()
    )

    # No debe existir una misma observación
    # contrato-periodo en entrenamiento y validación
    claves_train = set(zip(
        train["id_credito"], train["fecha_corte"]
    ))

    claves_val = set(zip(
        val["id_credito"], val["fecha_corte"]
    ))

    assert claves_train.isdisjoint(claves_val)

    registros_validacion.append({
        "fold": numero,
        "entrenamiento": len(train),
        "validacion": len(val),
        "corte_validacion": val["fecha_corte"].min().date(),
        "eventos": int(val[VARIABLE_OBJETIVO].sum())
    })

resumen_cv = pd.DataFrame(registros_validacion)

print("FINPREDICT | Preparación de GridSearchCV")
print("-" * 65)

display(resumen_cv)

print("-" * 65)
print("Tres folds temporales validados.")
print("Modelos base conservados sin modificaciones.")


FINPREDICT | Preparación de GridSearchCV
-----------------------------------------------------------------


,fold,entrenamiento,validacion,corte_validacion,eventos
0,1,210863,33792,2024-12-31,1848
1,2,242453,36111,2025-03-31,2116
2,3,276245,38164,2025-06-30,2252


-----------------------------------------------------------------
Tres folds temporales validados.
Modelos base conservados sin modificaciones.



### 9. Optimización de regresión logística

Se utiliza `GridSearchCV` para seleccionar hiperparámetros de regresión logística mediante tres particiones temporales expansivas.

**Objetivo de optimización:** maximizar Average Precision (PR-AUC), una métrica adecuada para evaluar la capacidad de ordenamiento de observaciones positivas en un contexto de clases desbalanceadas.

Se comparan distintos niveles de regularización y estrategias de ponderación de clases.

El preprocesamiento forma parte del Pipeline, por lo que la imputación y el escalado se ajustan exclusivamente con los datos de entrenamiento de cada fold.

La evaluación final permanece reservada y no interviene en la selección de hiperparámetros.


In [8]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 9
# GridSearchCV | Regresión logística
# ============================================================

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    make_scorer,
    average_precision_score,
    recall_score,
    f1_score
)

import pandas as pd
import time

# Búsqueda acotada y reproducible
parametros_logistica = {
    "modelo__C": [0.1, 1.0, 10.0],
    "modelo__class_weight": [None, "balanced"]
}

# Métricas de evaluación
metricas_cv = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "f1": "f1"
}

busqueda_logistica = GridSearchCV(
    estimator=pipeline_logistico,
    param_grid=parametros_logistica,
    scoring=metricas_cv,
    refit="pr_auc",
    cv=particiones,
    n_jobs=2,
    verbose=1,
    return_train_score=False,
    error_score="raise"
)

inicio = time.perf_counter()

busqueda_logistica.fit(X, y)

tiempo_busqueda_logistica = time.perf_counter() - inicio

# Resultados de experimentación
resultados_logistica_cv = pd.DataFrame(
    busqueda_logistica.cv_results_
)

columnas_resultados = [
    "param_modelo__C",
    "param_modelo__class_weight",
    "mean_test_pr_auc",
    "mean_test_roc_auc",
    "mean_test_recall",
    "mean_test_f1",
    "rank_test_pr_auc"
]

resumen_grid_logistica = (
    resultados_logistica_cv[columnas_resultados]
    .sort_values("rank_test_pr_auc")
    .reset_index(drop=True)
)

print("FINPREDICT | GridSearchCV - Regresión logística")
print("-" * 65)

display(resumen_grid_logistica.round(4))

print("\nMejores hiperparámetros:")
print(busqueda_logistica.best_params_)

print(f"\nMejor PR-AUC promedio: {busqueda_logistica.best_score_:.4f}")

print(
    f"Tiempo total de búsqueda: "
    f"{tiempo_busqueda_logistica:.2f} segundos"
)

print("-" * 65)
print("Optimización de regresión logística completada.")


Fitting 3 folds for each of 6 candidates, totalling 18 fits
FINPREDICT | GridSearchCV - Regresión logística
-----------------------------------------------------------------


,param_modelo__C,param_modelo__class_weight,mean_test_pr_auc,mean_test_roc_auc,mean_test_recall,mean_test_f1,rank_test_pr_auc
0,0.1,None,0.8567,0.9844,0.6661,0.7452,1
1,1.0,None,0.8566,0.9844,0.6659,0.7446,2
2,10.0,None,0.8566,0.9844,0.6659,0.7446,3
3,0.1,balanced,0.8492,0.9846,0.9290,0.6807,4
4,1.0,balanced,0.8491,0.9846,0.9290,0.6807,5
5,10.0,balanced,0.8491,0.9846,0.9290,0.6807,6



Mejores hiperparámetros:
{'modelo__C': 0.1, 'modelo__class_weight': None}

Mejor PR-AUC promedio: 0.8567
Tiempo total de búsqueda: 18.83 segundos
-----------------------------------------------------------------
Optimización de regresión logística completada.



### 10. Optimización de Random Forest

Se optimiza el modelo Random Forest mediante GridSearchCV con validación temporal expansiva.

La búsqueda evalúa combinaciones de profundidad máxima y número mínimo de observaciones por hoja, manteniendo constante el número de árboles para controlar el costo computacional.

La métrica principal de selección es Average Precision (PR-AUC). Se registran adicionalmente ROC-AUC, recall y F1-score.

Los resultados se compararán posteriormente con los de regresión logística y se documentarán mediante MLflow.


In [9]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 10
# GridSearchCV | Random Forest
# ============================================================

from sklearn.model_selection import GridSearchCV
import pandas as pd
import time

# Búsqueda acotada para CPU
parametros_rf = {
    "modelo__max_depth": [10, 14],
    "modelo__min_samples_leaf": [10, 30]
}

busqueda_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=parametros_rf,
    scoring=metricas_cv,
    refit="pr_auc",
    cv=particiones,
    n_jobs=2,
    verbose=1,
    return_train_score=False,
    error_score="raise"
)

inicio = time.perf_counter()

busqueda_rf.fit(X, y)

tiempo_busqueda_rf = time.perf_counter() - inicio

# Consolidar resultados
resultados_rf_cv = pd.DataFrame(
    busqueda_rf.cv_results_
)

columnas_rf = [
    "param_modelo__max_depth",
    "param_modelo__min_samples_leaf",
    "mean_test_pr_auc",
    "mean_test_roc_auc",
    "mean_test_recall",
    "mean_test_f1",
    "rank_test_pr_auc"
]

resumen_grid_rf = (
    resultados_rf_cv[columnas_rf]
    .sort_values("rank_test_pr_auc")
    .reset_index(drop=True)
)

print("FINPREDICT | GridSearchCV - Random Forest")
print("-" * 65)

display(resumen_grid_rf.round(4))

print("\nMejores hiperparámetros:")
print(busqueda_rf.best_params_)

print(f"\nMejor PR-AUC promedio: {busqueda_rf.best_score_:.4f}")

print(
    f"Tiempo total de búsqueda: "
    f"{tiempo_busqueda_rf:.2f} segundos"
)

print("-" * 65)
print("Optimización de Random Forest completada.")


Fitting 3 folds for each of 4 candidates, totalling 12 fits
FINPREDICT | GridSearchCV - Random Forest
-----------------------------------------------------------------


,param_modelo__max_depth,param_modelo__min_samples_leaf,mean_test_pr_auc,mean_test_roc_auc,mean_test_recall,mean_test_f1,rank_test_pr_auc
0,14,30,0.8631,0.9855,0.9284,0.6843,1
1,10,30,0.8623,0.9856,0.9307,0.6722,2
2,14,10,0.8618,0.9855,0.9145,0.7020,3
3,10,10,0.8609,0.9855,0.9278,0.6766,4



Mejores hiperparámetros:
{'modelo__max_depth': 14, 'modelo__min_samples_leaf': 30}

Mejor PR-AUC promedio: 0.8631
Tiempo total de búsqueda: 1069.77 segundos
-----------------------------------------------------------------
Optimización de Random Forest completada.
